In [ ]:
import zipfile
with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
  zip_ref.extractall('data')


In [ ]:
# import the dataset
import pandas as pd
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [ ]:
print('The dimensions of movies dataframe are:', movies_df.shape, '\nThe dimensions of ratings dataframe are:', ratings_df.shape)
print('')

In [ ]:
# Take a look at movies dataframe
movies_df.head()


In [ ]:
# Take a look at ratings dataframe
ratings_df.head()

In [ ]:
# Movie ID to movie name mapping
movie_names = movies_df.set_index('movieId')['title'].to_dict()
# Count unique users and movies
n_users = len(ratings_df.userId.unique())
n_items = len(ratings_df.movieId.unique())
print("Number of unique users:", n_users)
print("Number of unique movies:", n_items)
# Total matrix size
print("The full rating matrix will have:", n_users * n_items, "elements.")
print('---')
# Number of ratings
print("Number of ratings:", len(ratings_df))
print("Therefore: ", len(ratings_df) / (n_users*n_items) * 100, '% of the matrix is filled.')
print("We have an incredibly sparse matrix to work with here.")
print("And ... as you can imagine, as the number of users and products grow, the number of elements will increase by n*2")
print("You are going to need a lot of memory to work with global scale ... storing a full matrix in memory would be a challenge.")
print("One advantage here is that matrix factorization can realize the rating matrix implicitly, thus we don't need all the data")


In [ ]:
import torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader

# Creating the dataloader (necessary for PyTorch)

class Loader(Dataset):

    def __init__(self):
        self.ratings = ratings_df.copy()

        # Extract all user IDs and movie IDs
        users = ratings_df.userId.unique()
        movies = ratings_df.movieId.unique()

        # Producing new continuous IDs for users and movies
        self.userid2idx = {o: i for i, o in enumerate(users)}
        self.movieid2idx = {o: i for i, o in enumerate(movies)}

        # Reverse mappings
        self.idx2userid = {i: o for o, i in self.userid2idx.items()}
        self.idx2movieid = {i: o for o, i in self.movieid2idx.items()}

        # Replace original IDs with indexed IDs
        self.ratings.movieId = ratings_df.movieId.apply(
            lambda x: self.movieid2idx[x]
        )

        self.ratings.userId = ratings_df.userId.apply(
            lambda x: self.userid2idx[x]
        )

        # Features and targets
        self.x = self.ratings.drop(
            ['rating', 'timestamp'],
            axis=1
        ).values

        self.y = self.ratings['rating'].values

        # Transforms the data to tensors (ready for torch models.)
        self.x = torch.tensor(self.x)
        self.y = torch.tensor(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.ratings)

In [ ]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):

    def __init__(self, n_users, n_items, n_factors=20):
        super().__init__()

        # User embeddings
        self.user_factors = nn.Embedding(
            n_users,
            n_factors
        )

        # Movie embeddings
        self.item_factors = nn.Embedding(
            n_items,
            n_factors
        )

        # Initialize weights
        self.user_factors.weight.data.uniform_(0, 0.05)
        self.item_factors.weight.data.uniform_(0, 0.05)

    def forward(self, data):

        # Extract user and movie IDs
        users = data[:, 0]
        items = data[:, 1]

        # Get embeddings
        user_embedding = self.user_factors(users)
        item_embedding = self.item_factors(items)

        # Dot product
        predictions = (
            user_embedding * item_embedding
        ).sum(1)

        return predictions

In [ ]:
num_epochs = 128

# Check if GPU is available
cuda = torch.cuda.is_available()

print("Is running on GPU:", cuda)

# Initialize model
model = MatrixFactorization(
    n_users,
    n_items,
    n_factors=8
)

print(model)

# Print model parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

# Move model to GPU if available
if cuda:
    model = model.cuda()

# MSE Loss
loss_fn = torch.nn.MSELoss()

# Adam Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Training data
train_set = Loader()

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True
)

In [ ]:
from tqdm.notebook import tqdm

# Training loop
for it in tqdm(range(num_epochs)):

    losses = []

    for x, y in train_loader:

        # Move data to GPU
        if cuda:
            x = x.cuda()
            y = y.cuda()

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(x)

        # Calculate loss
        loss = loss_fn(
            outputs.squeeze(),
            y.type(torch.float32)
        )

        # Store loss
        losses.append(loss.item())

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    # Print average loss per epoch
    print(
        "iter #{}".format(it),
        "Loss:",
        sum(losses) / len(losses)
    )

In [ ]:
# By training the model, we will have tuned latent factors for movies and users.

c = 0
uw = 0
iw = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        print(name, param.data)

        if c == 0:
            uw = param.data
            c += 1

        else:
            iw = param.data

        # print('param_data', param.data)

In [ ]:
trained_movie_embeddings = model.item_factors.weight.data.cpu().numpy()

In [ ]:
trained_movie_embeddings # unique movie factor weights

In [ ]:
from sklearn.cluster import KMeans
# Fit the clusters based on the movie weights
kmeans = KMeans(n_clusters=10, random_state=0).fit(trained_movie_embeddings)

In [ ]:
"""
It can be seen here that the movies in the same cluster tend to have
similar genres. Also note that the algorithm is unfamiliar with the movie names
and only obtained the relationships by looking at the numbers
representing how users responded to movie selections.
"""

import numpy as np

for cluster in range(10):

    print(f"\nCluster #{cluster}")

    movs = []

    for movidx in np.where(kmeans.labels_ == cluster)[0]:

        movid = train_set.idx2movieid[movidx]

        rat_count = ratings_df.loc[
            ratings_df['movieId'] == movid
        ].shape[0]

        movs.append(
            (movie_names[movid], rat_count)
        )

    for mov in sorted(
        movs,
        key=lambda tup: tup[1],
        reverse=True
    )[:10]:

        print("\t", mov[0])